# Set Models for Classification
## Yoav Ram

This notebook compares three architectures for classifying **unordered sets** of items:
1. **Flattened MLP** — concatenate all items into one vector
2. **Deep Sets** — shared per-item network with sum pooling
3. **Set Transformer** — self-attention over items with mean pooling

We use the [Poker Hand dataset](https://archive.ics.uci.edu/dataset/158/poker+hand) as a running example: each hand is a set of 5 cards, and the task is to classify the hand type (pair, flush, straight, etc.).

The main goal is to understand how **inductive biases** — particularly **permutation invariance** — affect learning on set-structured data, and to motivate the move to sequence models in later notebooks.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt

import os
import jax
import jax.numpy as jnp
import numpy as np
import optax

print('jax', jax.__version__, jax.default_backend())

key = jax.random.key(2)

jax 0.5.3 cpu


## Problem Setup

A poker hand is a set of 5 cards. Each card has a **suit** (Hearts, Spades, Diamonds, Clubs) and a **rank** (Ace through King). The task is to classify the hand into one of 10 types:

| Class | Hand type       | Example             |
|-------|----------------|---------------------|
| 0     | Nothing         | 2♠ 5♦ 8♣ J♥ K♦     |
| 1     | One pair        | 3♠ 3♦ 7♣ J♥ K♦     |
| 2     | Two pairs       | 3♠ 3♦ 7♣ 7♥ K♦     |
| 3     | Three of a kind | 3♠ 3♦ 3♣ J♥ K♦     |
| 4     | Straight        | 5♠ 6♦ 7♣ 8♥ 9♦     |
| 5     | Flush           | 2♠ 5♠ 8♠ J♠ K♠     |
| 6     | Full house      | 3♠ 3♦ 3♣ 7♥ 7♦     |
| 7     | Four of a kind  | 3♠ 3♦ 3♣ 3♥ K♦     |
| 8     | Straight flush  | 5♠ 6♠ 7♠ 8♠ 9♠     |
| 9     | Royal flush     | 10♠ J♠ Q♠ K♠ A♠    |

This is a natural **set classification** problem: the order of cards in a hand does not matter — {3♠, 7♦, K♣, 2♥, 9♠} is the same hand regardless of how the cards are arranged.

## Data

The [Poker Hand dataset](https://archive.ics.uci.edu/dataset/158/poker+hand) from the UCI Machine Learning Repository contains over 1 million hands. Each hand is described by 10 attributes (suit and rank for each of 5 cards) and a class label.

In [3]:
raw_train = np.loadtxt(os.path.join('data', 'poker-hand-training-true.data'), delimiter=',', dtype=int)
raw_test = np.loadtxt(os.path.join('data', 'poker-hand-testing.data'), delimiter=',', dtype=int)
raw = np.concatenate([raw_train, raw_test], axis=0)

print(f"Total hands: {len(raw):,}")
print(f"Columns: {raw.shape[1]} (5 cards × 2 attributes + 1 label)")

Total hands: 1,025,010
Columns: 11 (5 cards × 2 attributes + 1 label)


In [4]:
# Columns: S1, C1, S2, C2, S3, C3, S4, C4, S5, C5, CLASS
suits = raw[:, [0, 2, 4, 6, 8]] - 1    # (N, 5), values 0-3
ranks = raw[:, [1, 3, 5, 7, 9]] - 1    # (N, 5), values 0-12
labels = raw[:, 10]                      # (N,),   values 0-9

hand_names = [
    'Nothing', 'One pair', 'Two pairs', 'Three of a kind', 'Straight',
    'Flush', 'Full house', 'Four of a kind', 'Straight flush', 'Royal flush'
]
n_classes = len(hand_names)

print("Class distribution:")
for c in range(n_classes):
    count = (labels == c).sum()
    print(f"  {c}: {hand_names[c]:<20s} {count:>8,d} ({100 * count / len(labels):.2f}%)")

Class distribution:
  0: Nothing               513,702 (50.12%)
  1: One pair              433,097 (42.25%)
  2: Two pairs              48,828 (4.76%)
  3: Three of a kind        21,634 (2.11%)
  4: Straight                3,978 (0.39%)
  5: Flush                   2,050 (0.20%)
  6: Full house              1,460 (0.14%)
  7: Four of a kind            236 (0.02%)
  8: Straight flush             17 (0.00%)
  9: Royal flush                 8 (0.00%)


In [5]:
# Shuffle and split: 80% train, 10% val, 10% test
rng = np.random.default_rng(172)
perm = rng.permutation(len(labels))
suits, ranks, labels = suits[perm], ranks[perm], labels[perm]

n = len(labels)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_suits  = jnp.array(suits[:n_train])
train_ranks  = jnp.array(ranks[:n_train])
train_labels = jnp.array(labels[:n_train])

val_suits  = jnp.array(suits[n_train:n_train + n_val])
val_ranks  = jnp.array(ranks[n_train:n_train + n_val])
val_labels = jnp.array(labels[n_train:n_train + n_val])

test_suits  = jnp.array(suits[n_train + n_val:])
test_ranks  = jnp.array(ranks[n_train + n_val:])
test_labels = jnp.array(labels[n_train + n_val:])

print(f"Train: {len(train_labels):,}  Val: {len(val_labels):,}  Test: {len(test_labels):,}")

Train: 820,008  Val: 102,501  Test: 102,501


## Shared Utilities

All three models share the same card embedding, loss function, and training loop.

Each card is identified by its suit (0–3) and rank (0–12), giving 4 × 13 = 52 unique cards. We represent each card as a one-hot vector of size 52 and project it through a learned embedding matrix to get a dense vector of size $d$:
$$\mathbf{e}_i = W_{\text{emb}} \cdot \text{onehot}(s_i \times 13 + r_i)$$
This is equivalent to a lookup in a $(52 \times d)$ embedding table — one learned vector per card.

In [ ]:
d_card = 16  # card embedding dimension
n_cards = 5
n_vocab = 52  # 4 suits × 13 ranks


def embed_cards(params, suits, ranks):
    """
    suits : (B, 5) int, values 0-3
    ranks : (B, 5) int, values 0-12
    Returns: (B, 5, d_card)
    """
    card_idx = suits * 13 + ranks          # (B, 5), values 0-51
    return params['card_emb'][card_idx]    # (B, 5, d_card)


def cross_entropy_loss(logits, targets):
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_probs[jnp.arange(targets.shape[0]), targets])


def count_params(pytree):
    leaves = jax.tree_util.tree_leaves(pytree)
    return sum(x.size for x in leaves if hasattr(x, 'size'))


def accuracy(params, model, suits, ranks, labels, batch_size=4096):
    """Compute accuracy in batches."""
    n = len(labels)
    correct = 0
    for i in range(0, n, batch_size):
        logits = model(params, suits[i:i + batch_size], ranks[i:i + batch_size])
        preds = jnp.argmax(logits, axis=-1)
        correct += jnp.sum(preds == labels[i:i + batch_size])
    return float(correct / n)

In [ ]:
def train(name, init_fn, model, n_steps=10_000, batch_size=256, lr=1e-3, seed=0):
    """Train a model and return params + training history."""
    key = jax.random.key(seed)
    key, subkey = jax.random.split(key)
    params = init_fn(subkey)
    print(f"{name}: {count_params(params):,} parameters")

    optimizer = optax.adamw(lr, weight_decay=1e-4)
    opt_state = optimizer.init(params)

    @jax.jit
    def step(params, opt_state, s, r, y):
        def loss_fn(p):
            return cross_entropy_loss(model(p, s, r), y)
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state_new = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), opt_state_new, loss

    train_losses = []
    val_log = []
    log_every = 1000
    n_train = len(train_labels)

    val_n = min(5000, len(val_labels))
    vs, vr, vy = val_suits[:val_n], val_ranks[:val_n], val_labels[:val_n]

    for i in range(1, n_steps + 1):
        key, subkey = jax.random.split(key)
        idx = jax.random.randint(subkey, (batch_size,), 0, n_train)
        params, opt_state, loss = step(
            params, opt_state, train_suits[idx], train_ranks[idx], train_labels[idx]
        )
        train_losses.append(float(loss))

        if i % log_every == 0:
            v_logits = model(params, vs, vr)
            v_loss = float(cross_entropy_loss(v_logits, vy))
            v_acc = float(jnp.mean(jnp.argmax(v_logits, axis=-1) == vy))
            val_log.append((i, v_loss, v_acc))
            print(f"  step {i:5d} | train loss={train_losses[-1]:.4f} | val loss={v_loss:.4f} | val acc={v_acc:.3f}")

    return params, train_losses, val_log

In [ ]:
def plot_training(name, train_losses, val_log):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(train_losses, alpha=0.3)
    if val_log:
        steps, vl, va = zip(*val_log)
        axes[0].plot(steps, vl, 'o-', label='val')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Cross-entropy loss')
    axes[0].set_title(f'{name} — Loss'); axes[0].legend()

    if val_log:
        axes[1].plot(steps, [100 * a for a in va], 'o-', color='green')
        axes[1].set_xlabel('Step'); axes[1].set_ylabel('Accuracy (%)')
        axes[1].set_title(f'{name} — Val Accuracy')

    plt.tight_layout(); plt.show()

# Model 1: Flattened MLP

The simplest approach: flatten all 5 card embeddings into a single vector and pass it through a standard MLP.

$$\mathbf{x} = [\mathbf{e}_1 \| \mathbf{e}_2 \| \cdots \| \mathbf{e}_5]$$
$$\hat{y} = \text{softmax}\big(W_2 \cdot \text{ReLU}(W_1 \mathbf{x} + b_1) + b_2\big)$$

This approach has two key limitations:
- **Fixed input size**: the model is hardcoded to exactly 5 cards.
- **Not permutation-invariant**: swapping two cards in the input changes the flattened vector and can change the output, even though the hand is the same. The model must learn to be approximately invariant from data alone.

In [ ]:
d_mlp = 128


def init_mlp(key):
    keys = jax.random.split(key, 3)
    scale = 0.02
    return {
        'card_emb': jax.random.normal(keys[0], (n_vocab, d_card)) * scale,
        'W1': jax.random.normal(keys[1], (n_cards * d_card, d_mlp)) * scale,
        'b1': jnp.zeros(d_mlp),
        'W2': jax.random.normal(keys[2], (d_mlp, n_classes)) * scale,
        'b2': jnp.zeros(n_classes),
    }


def mlp(params, suits, ranks):
    x = embed_cards(params, suits, ranks)     # (B, 5, d_card)
    x = x.reshape(x.shape[0], -1)             # (B, 5 * d_card)
    x = jax.nn.relu(x @ params['W1'] + params['b1'])
    return x @ params['W2'] + params['b2']

In [ ]:
%%time
mlp_params, mlp_losses, mlp_val = train('Flattened MLP', init_mlp, mlp)
plot_training('Flattened MLP', mlp_losses, mlp_val)

### Permutation sensitivity

Since the MLP concatenates cards in a fixed order, permuting the cards changes the input vector and can change the prediction. We verify this empirically: take a batch of hands, randomly shuffle the card order, and check how many predictions change.

In [ ]:
key, subkey = jax.random.split(key)
idx = jax.random.randint(subkey, (1000,), 0, len(test_labels))
s_orig, r_orig = test_suits[idx], test_ranks[idx]

key, subkey = jax.random.split(key)
perm = jax.random.permutation(subkey, 5)
s_perm, r_perm = s_orig[:, perm], r_orig[:, perm]

pred_orig = jnp.argmax(mlp(mlp_params, s_orig, r_orig), axis=-1)
pred_perm = jnp.argmax(mlp(mlp_params, s_perm, r_perm), axis=-1)
changed = float(jnp.mean(pred_orig != pred_perm))
print(f"MLP: {100 * changed:.1f}% of predictions changed after permuting card order")

# Model 2: Deep Sets

[Deep Sets](https://arxiv.org/abs/1703.06114) (Zaheer et al., 2017) is the simplest architecture that is **permutation-invariant by construction**.

Apply a shared network to each element independently, **sum** the results, then apply a second network:

$$z_i = \text{ReLU}(W_1 \mathbf{e}_i + b_1)$$
$$\hat{y} = \text{softmax}\!\left(W_3 \cdot \text{ReLU}\!\left(W_2 \sum_{i=1}^{n} z_i + b_2\right) + b_3\right)$$

Since addition is commutative, the output is the same regardless of the order of inputs.

$W_1$ is shared across all cards — it transforms each card independently. After sum-pooling, $W_2$ and $W_3$ classify the aggregated representation. Zaheer et al. proved that any permutation-invariant function on a finite set can be decomposed into this form, making Deep Sets a universal approximator for set functions.

In [ ]:
d_ds = 96


def init_deep_sets(key):
    keys = jax.random.split(key, 4)
    scale = 0.02
    return {
        'card_emb': jax.random.normal(keys[0], (n_vocab, d_card)) * scale,
        # Per-card transformation
        'W1': jax.random.normal(keys[1], (d_card, d_ds)) * scale,
        'b1': jnp.zeros(d_ds),
        # Post-pool classifier
        'W2': jax.random.normal(keys[2], (d_ds, d_ds)) * scale,
        'b2': jnp.zeros(d_ds),
        'W3': jax.random.normal(keys[3], (d_ds, n_classes)) * scale,
        'b3': jnp.zeros(n_classes),
    }


def deep_sets(params, suits, ranks):
    x = embed_cards(params, suits, ranks)                   # (B, 5, d_card)
    x = jax.nn.relu(x @ params['W1'] + params['b1'])       # (B, 5, d_ds)
    x = x.sum(axis=1)                                       # (B, d_ds)
    x = jax.nn.relu(x @ params['W2'] + params['b2'])       # (B, d_ds)
    return x @ params['W3'] + params['b3']                  # (B, n_classes)

In [ ]:
%%time
ds_params, ds_losses, ds_val = train('Deep Sets', init_deep_sets, deep_sets)
plot_training('Deep Sets', ds_losses, ds_val)

In [ ]:
pred_orig = jnp.argmax(deep_sets(ds_params, s_orig, r_orig), axis=-1)
pred_perm = jnp.argmax(deep_sets(ds_params, s_perm, r_perm), axis=-1)
changed = float(jnp.mean(pred_orig != pred_perm))
print(f"Deep Sets: {100 * changed:.1f}% of predictions changed after permuting card order")

# Model 3: Set Transformer

The [Set Transformer](https://arxiv.org/abs/1810.00825) (Lee et al., 2019) extends Deep Sets by adding **self-attention** between set elements before pooling. This allows the model to capture **interactions between cards** — for example, noticing that two cards share the same suit (relevant for flushes) or that ranks form a consecutive sequence (relevant for straights).

## Self-attention (without position)

Each card attends to every other card:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Since there is no positional encoding and no causal mask, the attention is **permutation-equivariant**: permuting the input cards produces the same permutation in the output. After mean-pooling over cards, the full model is **permutation-invariant**.

The transformer block uses pre-norm (layer normalization before each sub-layer) with residual connections. Layer normalization stabilizes training by normalizing activations — this is standard in transformers but not needed for the simpler MLP and Deep Sets architectures above.

In [ ]:
d_model = 32
n_heads = 2
d_ff = 64


def layer_norm(x, eps=1e-6):
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean((x - mean) ** 2, axis=-1, keepdims=True)
    return (x - mean) / jnp.sqrt(var + eps)


def self_attention(x, W_q, W_k, W_v, W_o, n_heads):
    B, T, D = x.shape
    d_head = D // n_heads
    Q = (x @ W_q).reshape(B, T, n_heads, d_head).transpose(0, 2, 1, 3)
    K = (x @ W_k).reshape(B, T, n_heads, d_head).transpose(0, 2, 1, 3)
    V = (x @ W_v).reshape(B, T, n_heads, d_head).transpose(0, 2, 1, 3)
    w = (Q @ K.transpose(0, 1, 3, 2)) / jnp.sqrt(d_head)
    w = jax.nn.softmax(w, axis=-1)
    z = (w @ V).transpose(0, 2, 1, 3).reshape(B, T, D)
    return z @ W_o


def init_set_transformer(key):
    keys = iter(jax.random.split(key, 20))
    scale = 0.02
    W = lambda shape: jax.random.normal(next(keys), shape) * scale
    return {
        'card_emb': W((n_vocab, d_card)),
        'proj':     W((d_card, d_model)),
        'W_q':      W((d_model, d_model)),
        'W_k':      W((d_model, d_model)),
        'W_v':      W((d_model, d_model)),
        'W_o':      W((d_model, d_model)),
        'ff_W1':    W((d_model, d_ff)),
        'ff_b1':    jnp.zeros(d_ff),
        'ff_W2':    W((d_ff, d_model)),
        'ff_b2':    jnp.zeros(d_model),
        'head_W':   W((d_model, n_classes)),
        'head_b':   jnp.zeros(n_classes),
    }


def set_transformer(params, suits, ranks):
    x = embed_cards(params, suits, ranks)       # (B, 5, d_card)
    x = x @ params['proj']                       # (B, 5, d_model)
    # Self-attention block with pre-norm + residual
    x_norm = layer_norm(x)
    x = x + self_attention(x_norm, params['W_q'], params['W_k'], params['W_v'], params['W_o'], n_heads)
    # FFN with pre-norm + residual
    x_norm = layer_norm(x)
    h = jax.nn.relu(x_norm @ params['ff_W1'] + params['ff_b1'])
    x = x + h @ params['ff_W2'] + params['ff_b2']
    # Pool and classify
    x = layer_norm(x)
    x = x.mean(axis=1)                          # (B, d_model)
    return x @ params['head_W'] + params['head_b']

In [ ]:
%%time
st_params, st_losses, st_val = train('Set Transformer', init_set_transformer, set_transformer)
plot_training('Set Transformer', st_losses, st_val)

In [ ]:
pred_orig = jnp.argmax(set_transformer(st_params, s_orig, r_orig), axis=-1)
pred_perm = jnp.argmax(set_transformer(st_params, s_perm, r_perm), axis=-1)
changed = float(jnp.mean(pred_orig != pred_perm))
print(f"Set Transformer: {100 * changed:.1f}% of predictions changed after permuting card order")

# Comparison

We compare all three models on the same train/val/test split: parameter count, training dynamics, and test accuracy.

In [ ]:
results = []
for name, params, model in [
    ('Flattened MLP', mlp_params, mlp),
    ('Deep Sets', ds_params, deep_sets),
    ('Set Transformer', st_params, set_transformer),
]:
    n_p = count_params(params)
    train_acc = accuracy(params, model, train_suits[:10000], train_ranks[:10000], train_labels[:10000])
    val_acc = accuracy(params, model, val_suits, val_ranks, val_labels)
    test_acc = accuracy(params, model, test_suits, test_ranks, test_labels)
    results.append((name, n_p, train_acc, val_acc, test_acc))

print(f"{'Model':<20s} {'Params':>8s} {'Train':>8s} {'Val':>8s} {'Test':>8s}")
print("-" * 55)
for name, n_p, tr, va, te in results:
    print(f"{name:<20s} {n_p:>8,d} {100 * tr:>7.1f}% {100 * va:>7.1f}% {100 * te:>7.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, losses, val_log in [
    ('Flattened MLP', mlp_losses, mlp_val),
    ('Deep Sets', ds_losses, ds_val),
    ('Set Transformer', st_losses, st_val),
]:
    axes[0].plot(losses, alpha=0.3, label=name)
    if val_log:
        steps, vl, va = zip(*val_log)
        axes[0].plot(steps, vl, 'o-', markersize=3)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].legend(); axes[0].set_title('Training loss')

names = [r[0] for r in results]
test_accs = [100 * r[4] for r in results]
bars = axes[1].bar(names, test_accs)
axes[1].set_ylabel('Test accuracy (%)')
axes[1].set_title('Test accuracy')
for bar, acc in zip(bars, test_accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout(); plt.show()

## Discussion

### What each architecture buys us

| Property | Flattened MLP | Deep Sets | Set Transformer |
|----------|:---:|:---:|:---:|
| Permutation invariant | ✗ | ✓ | ✓ |
| Variable-size input | ✗ | ✓ | ✓ |
| Inter-element interactions before pooling | ✗ | ✗ | ✓ |

- **Flattened MLP** is a useful baseline but has the wrong symmetry for sets. It must learn approximate invariance from data, wasting capacity. It also cannot generalise to different set sizes.

- **Deep Sets** provides the simplest correct symmetry for unordered inputs. For poker hands, where the class depends on counting how many cards share a rank or suit, sum pooling is a natural fit — it directly computes aggregate statistics. The theoretical result of Zaheer et al. guarantees that this architecture can represent *any* permutation-invariant function.

- **Set Transformer** adds self-attention before pooling, enabling content-dependent inter-card interactions. For example, attention can directly compare whether two cards share a suit (for flush detection) or have consecutive ranks (for straight detection), before the information is pooled into a single vector.

### Bridge to sequence models

In this notebook, the input is a **set** (unordered), so:
- No positional encoding is needed
- No causal masking is needed
- Permutation invariance is desired

In the sequence model notebooks ([RNN](RNN.ipynb), [GRU](GRU.ipynb), [transformer_text](transformer_text.ipynb), [nanochat](nanochat.ipynb)), the input is a **sequence** (ordered), so:
- **Positional encoding** injects order information (learned embeddings, or RoPE)
- **Causal masking** prevents attending to future tokens
- The model should be **permutation-sensitive** — "cat sat on mat" ≠ "mat sat on cat"

The attention mechanism is the same in both cases. The difference is entirely in what structural assumptions we add on top of it.

## References

1. **Zaheer et al. (2017)** Deep Sets. *NeurIPS*. [arXiv:1703.06114](https://arxiv.org/abs/1703.06114)
2. **Lee et al. (2019)** Set Transformer: A Framework for Attention-based Permutation-Invariant Input. *ICML*. [arXiv:1810.00825](https://arxiv.org/abs/1810.00825)
3. **Vaswani et al. (2017)** Attention Is All You Need. *NeurIPS*. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
4. **Poker Hand dataset.** UCI Machine Learning Repository. [Link](https://archive.ics.uci.edu/dataset/158/poker+hand)

# Colophon
This notebook was written by [Yoav Ram](http://python.yoavram.com).

This work is licensed under a [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) International License.

![Python logo](https://www.python.org/static/community_logos/python-logo.png)